In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Sample input DataFrame
data = pd.DataFrame({
    "text": [
        "The robot gazed at the stars, wondering about its creator.",
        "In the shimmering city of tomorrow, all was not well.",
        "Humanity's last hope was hidden in the ruins of the old world."
    ],
    "model": ["gpt-5", "llama-2", "gemini"],
    "temperature": [0.7, 1.0, 0.9],
    "batch_id": ["123", "123", "123"]
})

class LabellingBatch:
    text_area = widgets.Textarea(value='', description='Text:', layout=widgets.Layout(width="100%", height="500px"))
    usage_text = widgets.Combobox(value='', description='Usage:', layout=widgets.Layout(width="100%", height="20px"), ensure_option=False, options=['dialogue', 'exposition', 'opener', 'style'])
    model_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    temperature_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    batch_id_label = widgets.Label(value='', layout=widgets.Layout(width="100%"))
    rating = widgets.RadioButtons(
            options=["bad", "ok", "amazing"],
            value="bad",
            description='Rating:',
            layout=widgets.Layout(width="50%")
        )
    next_button = widgets.Button(description="Next", button_style='success')
    save_button = widgets.Button(description="Save labels", button_style='success')
    output = widgets.Output()

    input_df: pd.DataFrame
    outputs = []
    current_index: int
    
    def label_data(self, input_df: pd.DataFrame, experiment_name, current_index: int = 0):

        self.output_df: pd.DataFrame = pd.DataFrame()
        self.current_index = current_index
        self.input_df = input_df
        self.experiment_name = experiment_name
        
        # Attach event listener
        self.next_button.on_click(self.submit_and_next)
        self.save_button.on_click(self.write_labels)
        
        # Display initial data
        self.update_widgets(current_index)
        
        # Layout the widgets
        display(widgets.VBox([
            self.model_label,
            self.temperature_label,
            self.batch_id_label,
            self.text_area,
            self.usage_text,
            self.rating,
            self.next_button,
            self.save_button,
            self.output
        ]))

    def update_widgets(self, index: int):
        """Update widgets with the current row data."""
        df = self.input_df
        self.text_area.value = df.loc[index, "text"]
        self.usage_text.value = ""
        self.model_label.value = f"Model: {df.loc[index, 'model']}"
        self.temperature_label.value = f"Temperature: {df.loc[index, 'temperature']}"
        self.batch_id_label.value = f"Project: {df.loc[index, 'project_name']} Experiment: {df.loc[index, 'experiment_name']} Batch id: {df.loc[index, 'batch_id']}"
        self.rating.value = "bad"
    
    def submit_and_next(self, _):
        """Save current values and move to the next row."""
        new_row = {
            "target_text": self.text_area.value,
            "label": self.rating.value,
            "usage_text": self.usage_text.value,
            "input_index": self.current_index,
        }

        for column in ["text", "temperature", "model", "batch_id"]:
            new_row[column] = self.input_df.loc[self.current_index, column]

        self.outputs.append(new_row)
        
        # Auto-save every 10 entries
        if len(self.outputs) % 10 == 0:
            self._auto_save()
        
        # Increment index
        self.current_index += 1
        
        # Check if we reached the end
        if self.current_index < len(self.input_df):
            self.update_widgets(self.current_index)
        else:
            # Auto-save when finished
            self._auto_save()
            with self.output:
                clear_output()
                print("All entries have been labeled! (Auto-saved)")
        
        # Clear previous messages
        with self.output:
            clear_output()
            print(f"Entry {self.current_index}/{len(self.input_df)} labeled.")

    def _auto_save(self):
        """Auto-save labels to disk."""
        if len(self.outputs) > 0:
            import os
            os.makedirs("labels/generate_writing", exist_ok=True)
            pd.DataFrame(self.outputs).to_parquet(f"labels/generate_writing/{self.experiment_name}.parquet")
    
    def write_labels(self, _):
        """Manually save labels to disk."""
        self._auto_save()
        with self.output:
            clear_output()
            print(f"Labels saved! ({len(self.outputs)} entries)")




In [58]:
import os

PROJECT_NAME = "adam_and_eve"
EXPERIMENT_NAME = "opening_5"

folder = f"generated_text/{PROJECT_NAME}/{EXPERIMENT_NAME}/"
sorted(os.listdir(folder))


['2026-01-04 13:20:30.299485', '2026-01-04 13:21:01.645164']

In [63]:
index = 2
with open(f"{folder}/{sorted(os.listdir(folder))[index]}/user_prompt.txt", 'r') as f:
    print(f.read()[:200])

The audience will have read a story prior to this. Rewrite this opening of the short-story in third-person. Use your own style. The lab paradise that has developed to become as wonderful and simple as


In [64]:
# Option 1: Label a single folder (original behavior)
# ts = sorted(os.listdir(folder))[index]
# input_df = pd.read_parquet(f"{folder}{ts}/output_file.parquet")
# labeller = LabellingBatch()
# labeller.label_data(input_df, PROJECT_NAME + "-" + EXPERIMENT_NAME + "-" + ts)

# Option 2: Label multiple folders at once
# Specify the range of indices you want to label
all_folders = sorted(os.listdir(folder))

start_index = index
end_index = len(all_folders) - 1  # inclusive, defaults to last folder

selected_folders = all_folders[start_index:end_index+1]

# Combine all dataframes from selected folders
dataframes = []
skipped_folders = []
for ts in selected_folders:
    parquet_path = f"{folder}{ts}/output_file.parquet"
    if os.path.exists(parquet_path):
        try:
            df = pd.read_parquet(parquet_path)
            dataframes.append(df)
            print(f"✓ Loaded {ts} ({len(df)} entries)")
        except Exception as e:
            print(f"✗ Error loading {ts}: {e}")
            skipped_folders.append(ts)
    else:
        print(f"✗ Skipped {ts} (no output_file.parquet)")
        skipped_folders.append(ts)

if len(dataframes) == 0:
    print("ERROR: No valid folders found with output_file.parquet files!")
else:
    input_df = pd.concat(dataframes, ignore_index=True)
    print(f"\nTotal entries to label: {len(input_df)}")
    if skipped_folders:
        print(f"Skipped folders: {len(skipped_folders)}")
    
    labeller = LabellingBatch()
    experiment_name = f"{PROJECT_NAME}-{EXPERIMENT_NAME}-batch_{start_index}_to_{end_index}"
    labeller.label_data(input_df, experiment_name)

✓ Loaded 2026-01-04 13:22:25.808972 (10 entries)
✗ Skipped 2026-01-04 13:23:09.439810 (no output_file.parquet)

Total entries to label: 10
Skipped folders: 1


In [7]:
len(labeller.outputs)


10

In [8]:
df = pd.read_parquet(f"labels/generate_writing/first_attempt.parquet")
df.size

80